# 22 — Training data for the network

**What this notebook does.** It prepares the data the network will be trained on, on Colab (A100, Drive mounted). It does not design or train the network. Everything is written to `data/processed/psma/netdata/`, described field by field in `netdata/README.md`.

**Compact targets.** In the cohort the kinetics are per Monte Carlo source (9 compartments) and per lesion, and the spatial distribution inside each source comes from the real PET. So no voxel target map is stored per draw. Each phantom is stored once:

- inputs: PET (decay-corrected %IA/mL) and CT;
- the compartment code map and the lesion instance map;
- the spatial weight of each voxel within its source, from the **same functions as the Monte Carlo sources** (`psma_mc.source_weights_v2`, `psma_mc.lesion_weights`: PET-weighted, rest of body with the 10 mm halo set to its median). They are imported, not reimplemented.

Per draw and scenario only a parameter table is stored, and `net_data.rebuild` turns it into voxel maps, in NumPy or in PyTorch.

**What stays apart.** The 47 test phantoms are processed and stored in their own files (`*_test.parquet`, `shards/test_*.npz`). No statistic of theirs is computed or shown here. Doses are not here: they come from notebook 21, and the format leaves a place for them (§8).

**About this notebook.** Training data for the network (`src/net_data.py`): 4 mm phantoms in shards, per-draw kinetic parameters, baselines.
- **Inputs:** phantoms, cohort, cohort doses.
- **Outputs:** `data/processed/psma/netdata/` (README there is the reference for the network code).
- **Run time:** about 26 min with the cache; the cache takes about 2 h per grid.
- **Environment:** .venv.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import json
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import net_data as N

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
OUT = N.NETDATA
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
GRID_MM = 4.0
RULE = "hybrid"
N_PROCESSES = 2
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#7b5cd6"]
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9, "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": "#b8b7ae"})
pd.set_option("display.width", 230, "display.max_columns", 40, "display.max_rows", 150)
print("commit", COMMIT)

## 1. Sources and spatial weights

For each phantom, on its native PET grid:

- each compartment of `psma_mc.COMPARTMENTS_V2` gets a code (1-9) and its unit-sum weights from `source_weights_v2(study, c, "pet")`;
- each lesion keeps its number from `lesions.nii.gz` (cohort lesion $j$ is lesion id $j+1$; volumes checked against the cohort table) and its unit-sum weights from `lesion_weights`; all lesions get code 10;
- sources are disjoint, so one weight map and one code map hold them all.

`lesion_weights` drops a lesion whose PET sums to zero: such a lesion would have TIA in the cohort and no Monte Carlo source. Here it gets uniform weights over its mask and the status `uniform_fallback`. The MC functions reload organs and PET at every call: a context manager reads each image once while they run (only I/O is cached, the functions are unchanged). The cache is resumable (`netdata/_cache/`), built on 2 processes because notebook 21 runs on the other cores.

In [ ]:
split68 = pd.read_csv(N.PROCESSED / "psma_68ga_split.csv")
order = [(s, "train") for s in sorted(split68[split68.split == "train"].study_id)] + [(s, "test") for s in sorted(split68[split68.split == "test"].study_id)]
N.build_cache(order, processes=N_PROCESSES)
missing = [s for s, _ in order for g in N.GRIDS_MM if not (N.CACHE / f"g{g:g}" / f"{s}.json").exists()]
stale = [s for s, _ in order for g in N.GRIDS_MM if "rescued" not in json.loads((N.CACHE / f"g{g:g}" / f"{s}.json").read_text())["sources"]]
print(f"phantoms: {len(order)} ({sum(sp == 'train' for _, sp in order)} training, {sum(sp == 'test' for _, sp in order)} test); missing in the cache: {len(missing)}; built before the rescue rule: {len(set(stale))}")

In [ ]:
train_ids = [s for s, sp in order if sp == "train"]


def train_sources(grid):
    rows = []
    for s in train_ids:
        info = json.loads((N.CACHE / f"g{grid:g}" / f"{s}.json").read_text())
        rows += [pd.DataFrame(info["sources"]), pd.DataFrame(info["others"])]
    return pd.concat(rows, ignore_index=True)


sources_by_grid = {g: train_sources(g) for g in N.GRIDS_MM}
s4 = sources_by_grid[GRID_MM]
main_rule = s4[s4.rule == RULE]
status = main_rule.groupby(main_rule.code == N.LESION_CODE).status.value_counts().unstack(fill_value=0)
status.index = ["compartments", "lesions"]
print("Source status on the training phantoms (weights from psma_mc):")
display(status)

## 2. Common grid and label rule

The PET comes on three grids: 2.73 x 2.73 x 3.27, 4.07 x 4.07 x 5.0 and 4.07 x 4.07 x 2.0 mm. Two isotropic candidates are compared, **3 mm and 4 mm**. Resampling is by exact overlap of voxel boxes, axis by axis: totals are conserved for weights ("sum"), and PET and CT are volume-weighted means. The grid is anchored at the corner of the native volume and cropped to the body box plus one voxel.

On a coarser grid a voxel can hold several sources, but the network needs one label per voxel (its $\lambda$ come from the label). Three rules:

- **volume**: the source with the largest volume in the voxel;
- **activity**: the source with the largest PET activity in the voxel;
- **hybrid**: volume, except where the activity winner is a lesion.

A voxel is in the body if at least half of its volume is body. A lesion that wins no voxel gets its voxel of largest weight (`rescued`), so no source vanishes.

**Loss by resampling.** For each source, *retained* is the share of its weight that falls in voxels labelled as that source. The rest lands in a neighbour's voxels. In the stored data each source is renormalised over its own voxels, so its total TIA is exact: what resampling loses is position, not activity. The lesion TIA "lost" below is therefore the TIA that would be assigned to the neighbours without renormalisation. **Memory**: stored arrays are PET, CT, weight (float16), compartment and organs (uint8), lesion (uint16), 10 bytes per voxel; the network input (PET, CT) in float32 is 8 bytes per voxel.

In [ ]:
def memory_table():
    rows = []
    for g in N.GRIDS_MM:
        for s in train_ids:
            meta = json.loads((N.CACHE / f"g{g:g}" / f"{s}.json").read_text())["meta"]
            n = int(np.prod(meta["shape"]))
            rows.append(dict(grid_mm=g, study_id=s, native=" x ".join(f"{x:.2f}" for x in meta["native_spacing_mm"]), voxels_M=n / 1e6, stored_MB=n * 10 / 1e6, network_input_float32_MB=n * 2 * 4 / 1e6, file_MB=(N.CACHE / f"g{g:g}" / f"{s}.npz").stat().st_size / 1e6))
    return pd.DataFrame(rows)


memory = memory_table()
display(memory.groupby(["grid_mm", "native"]).agg(n=("study_id", "size"), voxels_M=("voxels_M", "median"), stored_MB=("stored_MB", "median"), stored_MB_max=("stored_MB", "max"), input_float32_MB=("network_input_float32_MB", "median")).round(2))
display(memory.groupby("grid_mm").agg(stored_MB_median=("stored_MB", "median"), stored_GB_training=("stored_MB", lambda x: x.sum() / 1e3)).round(2))

In [ ]:
params_train_all = N.kinetic_table("train")
lesion_tia = params_train_all[(params_train_all.arm == "main") & (params_train_all.scenario == "A") & (params_train_all.structure == "lesion")].groupby(["study_id", "source"]).tiac_h.mean()
BINS = [0, 0.5, 1, 2, 5, 10, 50, 1e5]
BIN_LABELS = ["<0.5", "0.5-1", "1-2", "2-5", "5-10", "10-50", ">=50"]


def lesion_loss(sources):
    les = sources[sources.code == N.LESION_CODE].copy()
    les["tiac_h"] = lesion_tia.reindex(pd.MultiIndex.from_arrays([les.study_id, les.source])).to_numpy()
    les["bin"] = pd.cut(les.native_volume_ml, BINS, labels=BIN_LABELS, right=False)
    les["lost_tiac"] = les.tiac_h * (1 - les.retained)
    return les


loss_rows, phantom_rows = [], []
for g, src in sources_by_grid.items():
    les = lesion_loss(src)
    for rule, d in les.groupby("rule"):
        t = d.groupby("bin", observed=True).agg(n=("retained", "size"), retained_median=("retained", "median"), retained_q10=("retained", lambda x: x.quantile(0.1)), share_below_half=("retained", lambda x: (x < 0.5).mean()), rescued=("rescued", "sum"))
        loss_rows.append(t.assign(grid_mm=g, rule=rule).reset_index())
        per = d.groupby("study_id").apply(lambda x: x.lost_tiac.sum() / x.tiac_h.sum(), include_groups=False)
        phantom_rows.append(dict(grid_mm=g, rule=rule, lesions=len(d), rescued=int(d.rescued.sum()), tia_weighted_loss_all=float(d.lost_tiac.sum() / d.tiac_h.sum()), phantom_loss_median=float(per.median()), phantom_loss_max=float(per.max())))
lesion_bins = pd.concat(loss_rows, ignore_index=True)
print("Lesion weight retained in the voxels labelled as that lesion, by native lesion volume (mL), training phantoms:")
display(lesion_bins.pivot_table(index="bin", columns=["grid_mm", "rule"], values="retained_median", observed=True).reindex(BIN_LABELS).round(3))
display(lesion_bins.pivot_table(index="bin", columns=["grid_mm", "rule"], values="n", observed=True).reindex(BIN_LABELS).iloc[:, :1].rename(columns=lambda c: "lesions"))
print("Share of lesion TIA (scenario A, mean over draws) outside the voxels labelled as its lesion, before renormalisation:")
grid_loss = pd.DataFrame(phantom_rows)
display(grid_loss.round(4))

In [ ]:
comp_rows = []
for g, src in sources_by_grid.items():
    comp = src[src.code != N.LESION_CODE]
    for (rule, source), d in comp.groupby(["rule", "source"]):
        comp_rows.append(dict(grid_mm=g, rule=rule, source=source, retained_median=d.retained.median(), retained_min=d.retained.min(), volume_ratio_median=(d.volume_ml / d.native_volume_ml).median()))
comp_table = pd.DataFrame(comp_rows)
print("Compartments, retained weight (median):")
display(comp_table.pivot_table(index="source", columns=["grid_mm", "rule"], values="retained_median").round(3))
print("Compartments, labelled volume on the grid over native volume (median):")
display(comp_table.pivot_table(index="source", columns=["grid_mm", "rule"], values="volume_ratio_median").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
centres = np.sqrt(np.array(BINS[:-1]).clip(0.25) * np.array(BINS[1:]).clip(max=200))
for ax, g in zip(axes, N.GRIDS_MM):
    for k, rule in enumerate(N.RULES):
        d = lesion_bins[(lesion_bins.grid_mm == g) & (lesion_bins.rule == rule)].set_index("bin").reindex(BIN_LABELS)
        ax.plot(centres, d.retained_median, "o-", color=COLORS[k], label=rule)
        ax.fill_between(centres, d.retained_q10, d.retained_median, color=COLORS[k], alpha=0.12, lw=0)
    ax.set_xscale("log")
    ax.set_title(f"{g:g} mm isotropic", loc="left")
    ax.set_xlabel("native lesion volume (mL)")
axes[0].set_ylabel("lesion weight retained (median, band to q10)")
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "lesion_retention_by_grid.png", dpi=150)

**Lesions on the 4 mm grid: dilation for evaluation.** The labels keep 93-94% of the weight of a lesion of 10 mL or more and 62% below 0.5 mL; in the stored data each lesion is renormalised over its own voxels, so its TIA is exact, but a network that follows the PET will put part of a small lesion into the neighbouring voxels. Lesion-level evaluation therefore does not use the 4 mm labels as they are: the predicted parameters are summarised per lesion on its voxels **dilated by r voxels** (ball), and the radius is the smallest that recovers the spill-out. For each lesion: share of its native weight (sum-resampled) inside the dilated voxels, and, for draw 0, scenario A, main arm, the TIA of the *other* sources rebuilt inside the dilated ring, as a share of the lesion TIA: what a plain sum over the dilated mask would add. Lesion doses are not computed on this grid at all: they come from the native grid with the per-lesion split kernel of nb21 (`cohort_doses.PhantomBasis.lesion_map`), from the per-lesion TIAC.


In [ ]:
import multiprocessing as mp

DILATION_CACHE = N.CACHE / f"lesion_dilation_g{GRID_MM:g}.parquet"
if DILATION_CACHE.exists():
    dilation = pd.read_parquet(DILATION_CACHE)
else:
    d0 = params_train_all[(params_train_all.draw == 0) & (params_train_all.arm == "main") & (params_train_all.scenario == "A")]
    jobs = [(s, GRID_MM, d0[d0.study_id == s]) for s in train_ids if json.loads((N.CACHE / f"g{GRID_MM:g}" / f"{s}.json").read_text())["meta"]["n_lesions"] > 0]
    with mp.get_context("spawn").Pool(4, maxtasksperchild=10) as pool:
        dilation = pd.concat(pool.imap_unordered(N.lesion_dilation, jobs), ignore_index=True)
    dilation.to_parquet(DILATION_CACHE, index=False)
vol4 = N.structure_volumes(main_rule)
dilation["native_volume_ml"] = vol4.reindex(pd.MultiIndex.from_arrays([dilation.study_id, dilation.source])).to_numpy()
dilation["bin"] = pd.cut(dilation.native_volume_ml, BINS, labels=BIN_LABELS, right=False)
mean_tia = lesion_tia.reindex(pd.MultiIndex.from_arrays([dilation.study_id, dilation.source])).to_numpy()
rows = []
for r in N.DILATION_RADII:
    lost = mean_tia * (1 - dilation[f"retained_r{r}"])
    rows.append(dict(r_vox=r, r_mm=r * GRID_MM, tia_weighted_loss=lost.sum() / mean_tia.sum(), retained_median=dilation[f"retained_r{r}"].median(), retained_q10=dilation[f"retained_r{r}"].quantile(0.1), share_retained_below_95pct=(dilation[f"retained_r{r}"] < 0.95).mean(),
                     voxels_over_own_median=(dilation[f"voxels_r{r}"] / dilation.n_voxels).median(), other_tia_over_lesion_median=(dilation[f"other_tia_r{r}"] / dilation.lesion_tia_h).median(), other_tia_over_lesion_weighted=dilation[f"other_tia_r{r}"].sum() / dilation.lesion_tia_h.sum()))
dilation_summary = pd.DataFrame(rows)
print(f"{len(dilation)} lesions, {dilation.study_id.nunique()} training phantoms, {GRID_MM:g} mm grid, rule {RULE}:")
display(dilation_summary.round(4))
display(dilation.groupby("bin", observed=True)[[f"retained_r{r}" for r in N.DILATION_RADII]].median().reindex(BIN_LABELS).round(3))
R_EVAL = int(dilation_summary[dilation_summary.tia_weighted_loss < 0.01].r_vox.min())
print(f"Smallest dilation with TIA-weighted loss below 1%: r = {R_EVAL} voxel ({R_EVAL * GRID_MM:g} mm); loss {dilation_summary.tia_weighted_loss[R_EVAL]:.4f}, retained q10 {dilation_summary.retained_q10[R_EVAL]:.3f}")


**Decision: isotropic 4 mm, label rule `hybrid`.**

- **Size.** The training phantoms take 4.0 GB as stored arrays at 4 mm against 9.3 GB at 3 mm (median 2.2 against 5.0 M voxels per phantom); the network input in float32 is a median 17 MB per phantom against 40 MB. float16 storage costs at most 0.2% on the PET, 0.5 HU on the CT, 0.05% on the weights.
- **Price, on small lesions only.** At 4 mm the labels keep a median 62% of the weight of a lesion below 0.5 mL (836 lesions, the largest volume class), 84-85% at 0.5-2 mL and 92-94% at 10 mL or more; 7.2% of the lesion TIA falls outside its own labels (3 mm: 5.5%). Compartments keep 83% (salivary glands) to 98% (liver). Each source is renormalised over its own voxels, so no TIA is lost, only its position.
- **Rule.** `hybrid` and `activity` keep the same lesion weight (7.2% lost at 4 mm), `volume` loses twice as much (14.2%) and has to rescue 224 lesions against 120. `hybrid` keeps the labelled organ volumes at 0.95-1.00 of the native volumes, while `activity` inflates the kidneys by 10% and the glands by 8%. So `hybrid`.
- **Consequence for evaluation.** Lesions are not evaluated on the 4 mm labels as they are. A 1-voxel (4 mm) dilation recovers the spill-out: 0.24% of the lesion TIA remains outside, 10th percentile of retained weight 0.958, and 2 voxels recover everything. The ring adds other sources' TIA, a median 7.9% of the lesion TIA (3.6% TIA-weighted), so the target is summarised on the same dilated mask. For the dose, that TIA is removed by local background subtraction (section 7: residual bias +1.7% median, +0.9% TIA-weighted, against +6.7% for the plain sum). Lesion doses are computed on the native grid with the per-lesion split kernel of nb21. The rule is written in the README.

In [ ]:
errors = pd.DataFrame([dict(study_id=s, **json.loads((N.CACHE / f"g{GRID_MM:g}" / f"{s}.json").read_text())["float16"]) for s in train_ids])
print(f"float16 storage at {GRID_MM:g} mm, worst voxel per phantom, over the training phantoms:")
display(errors.drop(columns="study_id").describe(percentiles=[0.5, 0.95]).loc[["50%", "95%", "max"]].T.round(5))

## 3. Splits

Training is the 180 training phantoms of nb08. A validation set is carved out of them **by subject**, about 15%, stratified like nb08: lesion-free, or tertile of the subject's median TLV (nb08 edges 5.0 and 35.8 mL), times single or multiple studies (seed 20261110). The 47 test phantoms keep their nb08 assignment and are stored apart.

In [ ]:
subjects = N.split_validation(train_ids)
assignment = {s: subjects.split[subj] for s, subj in split68.set_index("study_id").subject.items() if s in train_ids}
assignment.update({s: "test" for s, sp in order if sp == "test"})
display(pd.crosstab(subjects.stratum, subjects.split, margins=True))
inventory = pd.read_csv(N.PROCESSED / "psma_inventory.csv").set_index("study_id")
tv = pd.DataFrame({"net_split": pd.Series(assignment)}).join(inventory[["subject", "scanner", "tlv_ml", "n_lesions", "dx_mm", "dz_mm"]])
tv_only = tv[tv.net_split != "test"]
display(tv_only.groupby("net_split").agg(studies=("subject", "size"), subjects=("subject", "nunique"), tlv_median=("tlv_ml", "median"), tlv_q75=("tlv_ml", lambda x: x.quantile(0.75)), lesion_free_studies=("n_lesions", lambda x: int((x == 0).sum()))).round(1))
display(pd.crosstab(tv_only.scanner, tv_only.net_split))
assert not set(tv_only[tv_only.net_split == "train"].subject) & set(tv_only[tv_only.net_split == "val"].subject)

## 4. Target parametrisation

The cohort gives per source: $A_0$, $\lambda_w$, $\lambda_u$ and the TIAC (nb20: $\lambda_w$ from the 24-168 h slope, $A_0$ from its intercept, $\lambda_u$ such that $A_0(1/\lambda_w - 1/\lambda_u)$ equals the TIAC; $\lambda_u = \infty$, *degenerate*, when the late exponential alone already exceeds the TIAC).

- **$\lambda_{bio} = \lambda_w - \lambda_{phys}$**, $\lambda_{phys}$ = ln 2 / 159.5 h, on the log scale. A floor of 1e-5 /h is used for the log only (`lambda_bio_floored`); the rebuild uses the unfloored value.
- **$\log A_0$**: $A_0$ is a fraction of the injected activity for the whole source; the voxel target is $A_0 w(x)$, so per mL it is $A_0 w(x)/V_{voxel}$.
- **Late TIA** $= A_0/\lambda_w$ (`tiac_late_h` of nb20), the integral of the 24-168 h exponential from 0 to infinity: the part a late SPECT updates.
- **Early TIA** $=$ TIAC $-$ late TIA (`tiac_early_h` of nb20), **signed**: positive when the curve has a fast early component the late exponential misses (then $\lambda_u$ is degenerate), negative when there is an uptake phase ($-A_0/\lambda_u$ for a finite $\lambda_u$).

The voxel TIA is rebuilt as late + early, each distributed by the source weights, so it closes on the TIAC by construction for every source.

**Does the uptake-washout curve reproduce the TIAC?** The curve TIA $A_0(1/\lambda_w - 1/\lambda_u)$ (with $1/\lambda_u = 0$ when degenerate) is compared with the TIAC of the PBPK model, per compartment and per lesion volume class. Where it does not, $(A_0, \lambda_w, \lambda_u)$ alone is not a valid target. Rule (`net_data.curve_validity`, main arm, scenario A, training part): median |curve TIA / TIAC − 1| **at most 5%: `curve`**, targets $A_0$, $\lambda_{bio}$, $\lambda_u$ (where finite) and the early TIA; **5-20%: `late_early`**, targets $A_0$, $\lambda_{bio}$ and the early TIA, $\lambda_u$ dropped; **above 20%: `tia`**, the TIAC is the primary target, with its early and late parts. The 20% split is our choice: below it the late exponential carries most of the TIA and stays the kinetic prior for the Bayesian update; above it the early part is a large share, by voiding (bladder) or a multi-exponential blood clearance (blood pool, spine, rest of body, GI).


In [ ]:
params_train_all["net_split"] = params_train_all.study_id.map(assignment)
params_test = N.kinetic_table("test")
params_test["net_split"] = "test"


def sources_of(split_ids):
    return pd.concat([pd.DataFrame(json.loads((N.CACHE / f"g{GRID_MM:g}" / f"{s}.json").read_text())["sources"]) for s in split_ids])


test_ids = [s for s, sp in order if sp == "test"]
sources_train, sources_test = sources_of(train_ids), sources_of(test_ids)
params_train_all = N.mark_sources(params_train_all, sources_train)
params_test = N.mark_sources(params_test, sources_test)
validity = N.curve_validity(params_train_all[params_train_all.net_split == "train"])
params_train_all = N.add_target_kind(params_train_all, validity)
params_test = N.add_target_kind(params_test, validity)
COLUMNS = ["net_split", "split", "arm", "virtual_id", "study_id", "draw", "scenario", "source", "structure", "target_kind", "has_source", "a0", "lw", "lu", "tiac_h", "tiac_early_h", "tiac_late_h", "lambda_bio", "lambda_bio_floored", "lu_degenerate", "lu_target", "log_a0", "log_lambda_bio", "log_lambda_u", "log_tiac", "log_tiac_late", "early_share", "tiac_curve_h", "curve_error"]
params_train_all[COLUMNS].to_parquet(OUT / "params.parquet", index=False)
params_test[COLUMNS].to_parquet(OUT / "params_test.parquet", index=False)
print(f"params.parquet {len(params_train_all)} rows, params_test.parquet {len(params_test)} rows (test: written only)")
print("lambda_bio below the floor (training phantoms, all arms and scenarios):", int(params_train_all.lambda_bio_floored.sum()), "of", len(params_train_all), "; min", params_train_all.lambda_bio.min())
no_source = params_train_all[~params_train_all.has_source].drop_duplicates(["study_id", "source"])[["study_id", "source"]]
print("Sources with a cohort TIAC and no voxels (excluded from rebuild, baselines, evaluate):", no_source.to_records(index=False).tolist(), "; test:", int((~params_test.has_source).sum()), "rows")


In [ ]:
CLASS_BINS = [0, 0.5, 1, 2, 5, 10, 50, 1e5]
CLASS_LABELS = ["<0.5", "0.5-1", "1-2", "2-5", "5-10", "10-50", ">=50"]
ORDER = ["kidneys_functional", "liver", "spleen", "salivary_glands", "bladder", "gi_tract", "blood_pool", "spine", "rest_of_body", "lesion"]
volumes_train = N.structure_volumes(sources_train)
cv = params_train_all[(params_train_all.net_split == "train") & params_train_all.has_source].copy()
cv["volume_ml"] = volumes_train.reindex(pd.MultiIndex.from_arrays([cv.study_id, cv.source])).to_numpy()
cv["class"] = np.where(cv.structure == "lesion", "lesion " + pd.cut(cv.volume_ml, CLASS_BINS, labels=CLASS_LABELS, right=False).astype(str), cv.structure)
CLASS_ORDER = ORDER[:-1] + [f"lesion {b}" for b in CLASS_LABELS]


def curve_summary(g):
    e = g.curve_error.abs()
    return pd.Series(dict(n=len(g), p_degenerate=g.lu_degenerate.mean(), median_abs=e.median(), q90_abs=e.quantile(0.9), max_abs=e.max(), share_above_5pct=(e > 0.05).mean(), tia_weighted=(g.tiac_h - g.tiac_curve_h).abs().sum() / g.tiac_h.sum(), early_share_median=g.early_share.median()))


curve_table = {arm: cv[(cv.arm == arm) & (cv.scenario == "A")].groupby("class").apply(curve_summary, include_groups=False).reindex(CLASS_ORDER) for arm in ("main", "control")}
print("Uptake-washout curve TIA against the PBPK TIAC, |curve / TIAC - 1|, training part, scenario A. Main arm:")
display(curve_table["main"].round(3))
print("Control arm:")
display(curve_table["control"].round(3))
print("Target kind per structure (main arm, scenario A, training part):")
display(validity.reindex(ORDER).round(4))


In [ ]:
d = params_train_all[(params_train_all.net_split == "train") & (params_train_all.arm == "main") & (params_train_all.scenario == "A") & params_train_all.has_source].copy()
d["log_a0_per_ml"] = d.log_a0 - np.log(volumes_train.reindex(pd.MultiIndex.from_arrays([d.study_id, d.source])).to_numpy())


def summary(g):
    lb = g.lambda_bio
    within = g.groupby(["study_id", "source"]).log_lambda_bio.std()
    within_a0 = g.groupby(["study_id", "source"]).log_a0.std()
    return pd.Series(dict(
        target_kind=g.target_kind.iloc[0], n=len(g), log_a0_per_ml_mean=g.log_a0_per_ml.mean(), log_a0_per_ml_sd=g.log_a0_per_ml.std(), log_a0_skew=stats.skew(g.log_a0_per_ml), log_a0_sd_within=np.sqrt((within_a0 ** 2).mean()),
        T_bio_median_h=np.log(2) / lb.median(), T_eff_median_h=np.log(2) / g.lw.median(), log_lambda_bio_sd=g.log_lambda_bio.std(), log_lambda_bio_skew=stats.skew(g.log_lambda_bio), log_lambda_bio_sd_within=np.sqrt((within ** 2).mean()),
        share_T_eff_above_130h=(g.lw < np.log(2) / 130).mean(), p_lu_degenerate=g.lu_degenerate.mean(), lu_median=g.lu[g.lu_target].median() if g.lu_target.any() else np.nan, log_lambda_u_sd=g.log_lambda_u[g.lu_target].std() if g.lu_target.any() else np.nan,
        early_share_median=g.early_share.median(), early_share_q05=g.early_share.quantile(0.05), early_share_q95=g.early_share.quantile(0.95)))


dist = d.groupby("structure").apply(summary, include_groups=False).reindex(ORDER)
dist.to_csv(OUT / "target_distributions.csv")
print("Network targets, training part (validation excluded), main arm, scenario A. 'within' = RMS of the SD over the 10 draws of the same phantom (same PET).")
display(dist)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)
for ax, (col, label) in zip(axes, [("log_a0_per_ml", "log A0 per mL (fraction of IA / mL)"), ("log_lambda_bio", "log lambda_bio (1/h)"), ("log_lambda_u", "log lambda_u (1/h), lambda_u targets only")]):
    data = [d.loc[(d.structure == s) & (d.lu_target if col == "log_lambda_u" else True), col].dropna().to_numpy() for s in ORDER]
    keep = [k for k, x in enumerate(data) if len(x) > 20]
    ax.violinplot([data[k] for k in keep], positions=keep, vert=False, showmedians=True, widths=0.8)
    ax.set_xlabel(label)
axes[0].set_yticks(range(len(ORDER)), ORDER)
fig.tight_layout()
fig.savefig(FIG / "target_distributions.png", dpi=150)

**Reading the targets** (training part, main arm, scenario A):

- **Target kinds.** The uptake-washout curve reproduces the TIAC within 5% (median) only for **lesions** (2.1-3.1% median in every volume class, 2.9% pooled) and **salivary glands** (2.1%): kind `curve`. **Kidneys 6.0%, liver 7.9%, spleen 11.7%**: kind `late_early`. Their curves are almost always degenerate (87-94%), and the PBPK curve has a fast component before 24 h that the late exponential does not carry (early share median 6-12%, 95th percentile 13-24%). $A_0$ and $\lambda_{bio}$ still describe the late part, which is what a late SPECT updates, and the early TIA is the extra target. **Bladder 28%, GI 26%, blood pool 34%, spine 33%, rest of body 75%**: kind `tia`; the early part is too large for the late exponential to describe the source. These shares do not depend on lesion size, and the control arm gives the same picture (liver 11%, kidneys 2.4%, glands 0%).
- **$\log A_0$ per mL.** The SD between virtual patients is 0.35-0.92. A large part of it is spread between draws of the *same* PET (`within`): kidneys 0.37 of 0.41, liver 0.32 of 0.47, spleen 0.28 of 0.52, glands 0.50 of 0.73, lesions 0.72 of 0.92. That part is not predictable from the PET. It is the posterior spread of the nb18 fit carried into the cohort, so it bounds what a network can reach.
- **$\lambda_{bio}$.** The `within` SD equals the total SD for every structure (kidneys 0.17, liver and spleen 0.32, glands 0.15, lesions 0.60). In scenario A the washout does not depend on the phantom, hence not on the PET (nb12: T_eff set only by $\lambda_{rel}$). A network can learn it from the PET only in scenario B, by construction. Median T_eff: kidneys 43 h, liver and spleen 30 h, glands 33 h, lesions 65 h. **T_eff above 130 h: 0.03% of lesions, none elsewhere.** The cohort has no heavy tail toward T_phys, unlike the Deep Blue lesions of nb02 and nb07. This is declared for the Bayesian update: a prior trained here will not put mass there.
- **$\lambda_u$** is a target for 23% of lesion rows and 36% of gland rows (finite), with log SD 1.3 and 1.1: a weak, masked target.
- **Blood pool and spine** have nearly the same late kinetics (median T_eff 6.5 h, same SD and skew): in the cohort the spine source follows the plasma. Both are `tia`, so this does not enter a kinetic prior.

## 5. Baselines in the same format

Two predictions per virtual patient and source (mean and SD; logs except the early share, which is linear), keyed like the parameter table (`arm`, `virtual_id`, `source`), so `net_data.evaluate` scores the network and the baselines with the same code. Quantities: log $A_0$, log $\lambda_{bio}$, log $\lambda_u$ (only on `curve` structures where finite), log TIAC, log late TIA, early share (early TIA / TIAC).

- **Population prior** (`baseline_population*.parquet`): no PET. Per arm and structure, mean and SD over the *training part* (validation excluded), scenario A, of log $A_0$, log TIAC and log late TIA per mL (multiplied back by the source volume of the phantom), log $\lambda_{bio}$, log $\lambda_u$, early share, and the share of degenerate $\lambda_u$.
- **PBPK fit from the real PET** (`baseline_pbpk_fit*.parquet`), main arm only: the posterior predictive of the phantom, i.e. the other 9 scenario-A draws of the same phantom (the target draw is left out, because it shares its posterior draw of the densities). This is the nb18 fit under the frozen population, which is how the cohort was generated. The same prediction applies to every scenario of that draw, since the fit does not know about a PET-washout link.

The second baseline is the generator itself: in scenario A no predictor from the PET can beat it in expectation, apart from noise from 9 draws. The control arm has 2 draws per phantom, too few for this baseline: it has the population prior only.


In [ ]:
prior = N.population_prior(params_train_all, volumes_train, [s for s, v in assignment.items() if v == "train"])
prior.to_parquet(OUT / "population_prior.parquet", index=False)
display(prior.set_index(["arm", "structure"]).round(3))


volumes_test = N.structure_volumes(sources_test)
baselines = {"population": N.population_prediction(params_train_all, volumes_train, prior), "pbpk_fit": N.pbpk_leave_one_out(params_train_all)}
baselines_test = {"population": N.population_prediction(params_test, volumes_test, prior), "pbpk_fit": N.pbpk_leave_one_out(params_test)}
for name in baselines:
    baselines[name].assign(net_split=lambda x: x.study_id.map(assignment)).to_parquet(OUT / f"baseline_{name}.parquet", index=False)
    baselines_test[name].assign(net_split="test").to_parquet(OUT / f"baseline_{name}_test.parquet", index=False)
print({k: len(v) for k, v in baselines.items()}, "rows; test baselines written only")

In [ ]:
val = params_train_all[(params_train_all.net_split == "val") & (params_train_all.arm == "main")]
evaluation = pd.concat([N.evaluate(b, val, by=("structure",), scenarios=["A"]).assign(baseline=name) for name, b in baselines.items()])
print("Validation phantoms, main arm, scenario A: the same evaluation code the network will use.")
display(evaluation.pivot_table(index=["quantity", "structure"], columns="baseline", values=["median_abs_error", "coverage_68", "coverage_95"]).round(3))

**Reading the baselines** (27 validation phantoms, main arm, scenario A):

- **The PBPK fit beats the population where nb18 said it would.** Median absolute error on log $A_0$: liver 0.21 against 0.40, spleen 0.17 against 0.37, glands 0.36 against 0.62, bladder 0.12 against 0.52, lesions 0.55 against 0.66. **Kidneys 0.27 against 0.27: no gain**, as in nb18. The same pattern holds on log TIAC and log late TIA.
- **No gain on $\lambda_{bio}$:** kidneys 0.110 against 0.110, liver 0.224 against 0.220, lesions 0.398 against 0.391. In scenario A the PET carries no washout information (section 4), so this is the reference a network must match in A and beat in B.
- **The early share** is predicted to 0.02-0.07 by the fit and 0.03-0.14 by the population. Larger errors are on the `tia` sources (bladder, blood pool, spine).
- **Predictive intervals are Student t.** The PBPK baseline estimates a mean and an SD from the other 9 draws of the phantom. For a new draw the predictive interval is Student t with 8 degrees of freedom and scale SD·√(1 + 1/9) (`net_data.predictive_halfwidth`); `evaluate` uses it for every method that gives the number of draws (`<q>_n`). With it the PBPK baseline covers **0.926-0.978 at 95% and 0.652-0.726 at 68%** over all quantities and structures. With the Gaussian interval the first run gave 0.885-0.922 and 0.600-0.667, i.e. the 0.900 and 0.627 expected from 9 draws. The spread across structures is what 27 validation phantoms allow (10 correlated draws each). $\lambda_u$ is predicted from the finite draws only (52% of lesion rows have at least two), with as many degrees of freedom as draws: 0.926-0.934 at 95%.
- **The population prior** is estimated from over 300 rows per structure, so t and Gaussian are the same. It covers 0.90-1.00 at 95% and 0.57-0.85 at 68%. It is not calibrated per structure, being one SD pooled over phantoms (bladder log TIAC 1.00 at 95%).

## 6. Format for Colab

Arrays are packed into a few `.npz` shards (`netdata/shards/{train,val,test}_NN.npz`, 40 phantoms each, deflate). Each member is `<study_id>/<field>.npy`, so `np.load` reads one phantom without the others; `net_data.read_shard` returns `{study_id: {field: array}}`. HDF5 would also work, but `h5py` is not in `.venv` and the brief says not to install. Parameter tables are parquet. The copy rate from Drive to the Colab disk is not measured here: three rates are assumed.

In [ ]:
start = time.time()
where = N.pack_shards(assignment, GRID_MM)
print(f"packed in {time.time() - start:.0f} s")
phantom_rows, source_rows, source_rows_test = [], [], []
for s, net in assignment.items():
    info = json.loads((N.CACHE / f"g{GRID_MM:g}" / f"{s}.json").read_text())
    meta = info["meta"]
    row = dict(study_id=s, subject=inventory.subject[s], net_split=net, shard=where[s], scanner=inventory.scanner[s], grid_mm=GRID_MM, rule=RULE, shape=json.dumps(meta["shape"]), native_spacing_mm=json.dumps(meta["native_spacing_mm"]), native_box=json.dumps(meta["native_box"]), offset_vox=json.dumps(meta["offset_vox"]), affine=json.dumps(meta["affine"]), n_lesions=meta["n_lesions"], dose_available=False)
    if net != "test":
        row.update(tlv_ml=float(inventory.tlv_ml[s]), stratum=subjects.stratum[inventory.subject[s]])
    phantom_rows.append(row)
    (source_rows_test if net == "test" else source_rows).append(pd.DataFrame(info["sources"]).assign(net_split=net))
pd.DataFrame([r for r in phantom_rows if r["net_split"] != "test"]).to_parquet(OUT / "phantoms.parquet", index=False)
pd.DataFrame([r for r in phantom_rows if r["net_split"] == "test"]).to_parquet(OUT / "phantoms_test.parquet", index=False)
pd.concat(source_rows).to_parquet(OUT / "sources.parquet", index=False)
pd.concat(source_rows_test).to_parquet(OUT / "sources_test.parquet", index=False)
files = sorted(p for p in OUT.rglob("*") if p.is_file() and "_cache" not in p.parts)
sizes = pd.DataFrame([dict(file=str(p.relative_to(OUT)), MB=p.stat().st_size / 1e6) for p in files])
display(sizes.round(1))
total = sizes.MB.sum()
print(f"total {total / 1e3:.2f} GB; shards {sizes[sizes.file.str.startswith('shards')].MB.sum() / 1e3:.2f} GB")
for rate in (30, 60, 120):
    print(f"copy Drive -> Colab local disk at {rate} MB/s (assumed, not measured): {total / rate / 60:.1f} min")

In [ ]:
start = time.time()
shard = OUT / "shards" / "val_00.npz"
loaded = N.read_shard(shard)
seconds = time.time() - start
raw = sum(a.nbytes for f in loaded.values() for a in f.values())
print(f"read {shard.name}: {len(loaded)} phantoms, {shard.stat().st_size / 1e6:.0f} MB on disk, {raw / 1e6:.0f} MB in memory, {seconds:.1f} s here ({raw / 1e6 / seconds:.0f} MB/s decompressed)")

## 7. Checks

**Rebuild.** The shards are read back and the voxel TIA is rebuilt with `net_data.rebuild` for each training and validation phantom, main arm draws 0, 4 and 9 in scenarios A, B+0.6 and B−0.6, and the first control draw. The voxel TIA, and its late and early parts, are summed over each source and compared with the cohort (early as a share of the TIAC, since it can be near zero).

**Excluded and declared:** the bladder of `psma_3959d1c381a5bcd6_2015-11-12`. That PET is cropped at k ≥ 84 (exclusion list, nb11) and the bladder lies below the crop: the phantom has no bladder voxels and the full Monte Carlo run has no bladder source for it either (`skip_bladder`, "empty source"; 226 bladder sources for 227 phantoms). The cohort still gives it a TIAC. It has `has_source = False` in the parameter table and is left out of the rebuild, the baselines and `evaluate`; its TIAC is lost for that phantom (whole-body totals of this study were already not to be used).


In [ ]:
CHECK_DRAWS = (0, 4, 9)
CHECK_SCENARIOS = ("A", "B+0.6", "B-0.6")
sources_tv = pd.read_parquet(OUT / "sources.parquet")
check_rows, voxel_rows = [], []
for shard_name in sorted({where[s] for s, v in assignment.items() if v != "test"}):
    for s, arrays in N.read_shard(OUT / "shards" / shard_name).items():
        src = sources_tv[sources_tv.study_id == s]
        n_les = int(src.lesion_id.max()) if (src.code == N.LESION_CODE).any() else 0
        wc, wl = N.source_sum_tables(src, n_les)
        comp, les = arrays["compartment"], arrays["lesion"].astype(np.int64)
        key = np.where(les > 0, 1000 + les, comp.astype(np.int64)).ravel()
        structure_of_key = {N.COMPARTMENT_CODES[c]: c for c in N.SOURCE_COMPARTMENTS}
        p = params_train_all[(params_train_all.study_id == s) & params_train_all.has_source]
        picks = p[((p.arm == "main") & p.draw.isin(CHECK_DRAWS) & p.scenario.isin(CHECK_SCENARIOS)) | ((p.arm == "control") & (p.draw == 0))]
        for (arm, vid, scen), g in picks.groupby(["arm", "virtual_id", "scenario"]):
            ct, lt = N.parameter_arrays(g, n_les)
            r = N.rebuild(comp, les, arrays["weight"], wc, wl, ct, lt)
            sums = {part: np.bincount(key, weights=r[part].ravel(), minlength=1001 + n_les) for part in ("tia", "tia_late", "tia_early")}
            for x in g.itertuples(index=False):
                k = 1001 + int(x.source.split("_")[1]) if x.source.startswith("lesion_") else N.COMPARTMENT_CODES[x.source]
                check_rows.append(dict(study_id=s, arm=arm, scenario=scen, virtual_id=vid, source=x.source, structure=x.structure, target_kind=x.target_kind, tiac_h=x.tiac_h, tiac_late_h=x.tiac_late_h, tiac_early_h=x.tiac_early_h, rebuilt=sums["tia"][k], rebuilt_late=sums["tia_late"][k], rebuilt_early=sums["tia_early"][k]))
            if arm == "main" and scen == "A":
                for code, name in structure_of_key.items():
                    m = (comp == code) & (les == 0) & r["inside"]
                    if m.any():
                        voxel_rows.append(dict(study_id=s, virtual_id=vid, structure=name, voxels=int(m.sum()), degenerate_voxels=int(r["degenerate"][m].sum()), tia=float(r["tia"][m].sum()), tia_degenerate=float(r["tia"][m & r["degenerate"]].sum()), zero_weight_voxels=int((r["w"][m] == 0).sum())))
                m = (les > 0) & r["inside"]
                if m.any():
                    voxel_rows.append(dict(study_id=s, virtual_id=vid, structure="lesion", voxels=int(m.sum()), degenerate_voxels=int(r["degenerate"][m].sum()), tia=float(r["tia"][m].sum()), tia_degenerate=float(r["tia"][m & r["degenerate"]].sum()), zero_weight_voxels=int((r["w"][m] == 0).sum())))
check = pd.DataFrame(check_rows)
check["error"] = check.rebuilt / check.tiac_h - 1
check["error_late"] = check.rebuilt_late / check.tiac_late_h - 1
check["error_early_share"] = (check.rebuilt_early - check.tiac_early_h) / check.tiac_h
check.to_parquet(OUT / "figures" / "rebuild_check.parquet", index=False)
print(f"{check.study_id.nunique()} phantoms, {check.virtual_id.nunique()} draws, {len(check)} source x draw x scenario rows (sources without voxels excluded: {no_source.to_records(index=False).tolist()})")
display(check.groupby("structure").agg(target_kind=("target_kind", "first"), n=("error", "size"), max_abs_error=("error", lambda x: x.abs().max()), max_abs_error_late=("error_late", lambda x: x.abs().max()), max_abs_error_early_share=("error_early_share", lambda x: x.abs().max()), share_within_1pct=("error", lambda x: (x.abs() <= 0.01).mean())).reindex(ORDER))
ALL_OK = bool((check.error.abs() <= 0.01).all() and (check.error_late.abs() <= 0.01).all() and (check.error_early_share.abs() <= 0.01).all())
print("All sources within 1% (TIA, late, early share):", ALL_OK)


In [ ]:
voxels = pd.DataFrame(voxel_rows)
on_grid = voxels.groupby("structure")[["voxels", "degenerate_voxels", "tia", "tia_degenerate", "zero_weight_voxels"]].sum()
on_grid = pd.DataFrame({"voxel_share_degenerate": on_grid.degenerate_voxels / on_grid.voxels, "tia_share_degenerate": on_grid.tia_degenerate / on_grid.tia, "share_zero_weight": on_grid.zero_weight_voxels / on_grid.voxels}).reindex(ORDER)
on_grid["table_share_degenerate"] = d.groupby("structure").lu_degenerate.mean().reindex(ORDER)
print(f"Degenerate lambda_u on the {GRID_MM:g} mm grid, training and validation phantoms, main arm, scenario A, draws {CHECK_DRAWS}:")
display(on_grid.round(4))

**Single-lesion TIA for the dose.** The dilated-mask sum of section 2 also holds the TIA of the other sources in the ring, and that part must not enter the lesion dose. The TIA of lesion $i$ alone is obtained as (`net_data.separate_lesion`):

$$\mathrm{TIA}_i = \sum_{x \in D_i} T(x) \;-\; \sum_{x \in D_i \setminus L_i} b_{c(x)},$$

where $T$ is the voxel TIA map (predicted or true), $L_i$ the lesion's own voxels, $D_i$ its 1-voxel dilation without the voxels of other lesions, $c(x)$ the source label of a ring voxel and $b_c$ the local background of source $c$: the median of $T$ over the voxels of $c$ in a 1-voxel shell just outside $D_i$ (lesion voxels excluded; the source median over the window if $c$ is absent from the shell). A 1-voxel shell gave less bias than 2 or 3 voxels on three phantoms. The background is taken from the map itself, so the same rule applies to a prediction.

Tested on every training and validation phantom with lesions, draw 0, scenario A, main arm, on three maps:

- **label**: the target as stored (each lesion inside its own labels);
- **spill**: the same organs, with each lesion distributed by its native weight without renormalisation, so part of it falls in the neighbours' voxels, as a network that follows the PET would put it;
- **baseline**: the spill map of the PBPK-fit baseline (TIAC = exp of its predicted log mean, per source and lesion).

The separated value is compared with the true lesion TIAC (label, spill) and with the baseline's own lesion TIAC (baseline): the ratio is the bias of the separation alone. The last column shows the error of the baseline against the truth, for scale.


In [ ]:
SEPARATION_CACHE = N.CACHE / f"lesion_separation_g{GRID_MM:g}_r{R_EVAL}_s{N.SHELL_VOXELS}.parquet"
if SEPARATION_CACHE.exists():
    separation = pd.read_parquet(SEPARATION_CACHE)
else:
    d0 = params_train_all[(params_train_all.draw == 0) & (params_train_all.arm == "main") & (params_train_all.scenario == "A") & params_train_all.has_source]
    b0 = baselines["pbpk_fit"][baselines["pbpk_fit"].draw == 0]
    jobs = []
    for s in train_ids:
        if json.loads((N.CACHE / f"g{GRID_MM:g}" / f"{s}.json").read_text())["meta"]["n_lesions"] == 0:
            continue
        t = d0[d0.study_id == s]
        p = b0[b0.study_id == s]
        jobs.append((s, GRID_MM, R_EVAL, dict(zip(t.source, t.tiac_h)), dict(zip(p.source, np.exp(p.log_tiac_mean)))))
    with mp.get_context("spawn").Pool(4, maxtasksperchild=10) as pool:
        separation = pd.concat(pool.imap_unordered(N.lesion_separation, jobs), ignore_index=True)
    separation.to_parquet(SEPARATION_CACHE, index=False)
separation["native_volume_ml"] = volumes_train.reindex(pd.MultiIndex.from_arrays([separation.study_id, separation.source])).to_numpy()
separation["bin"] = pd.cut(separation.native_volume_ml, CLASS_BINS, labels=CLASS_LABELS, right=False)
separation["net_split"] = separation.study_id.map(assignment)
COMPARISONS = {"label, plain sum": ("label_plain", "tiac_true"), "label, separated": ("label_separated", "tiac_true"), "spill, plain sum": ("spill_plain", "tiac_true"), "spill, separated": ("spill_separated", "tiac_true"),
               "baseline, separated / its own TIAC": ("baseline_separated", "tiac_pred"), "baseline TIAC / true": ("tiac_pred", "tiac_true")}


def ratio_summary(d):
    out = {}
    for name, (a, b) in COMPARISONS.items():
        r = d[a] / d[b]
        out[name] = pd.Series(dict(median=r.median(), tia_weighted=d[a].sum() / d[b].sum(), q10=r.quantile(0.1), q90=r.quantile(0.9)))
    return pd.DataFrame(out).T


print(f"{len(separation)} lesions, {separation.study_id.nunique()} phantoms (training and validation), dilation r = {R_EVAL}, background shell {N.SHELL_VOXELS} voxel. Ratios:")
separation_summary = ratio_summary(separation)
display(separation_summary.round(4))
by_bin = pd.DataFrame({b: (g.spill_separated / g.tiac_true).median() for b, g in separation.groupby("bin", observed=True)}, index=["spill, separated / true (median)"]).T
by_bin["spill, plain / true (median)"] = separation.groupby("bin", observed=True).apply(lambda g: (g.spill_plain / g.tiac_true).median(), include_groups=False)
by_bin["baseline, separated / own (median)"] = separation.groupby("bin", observed=True).apply(lambda g: (g.baseline_separated / g.tiac_pred).median(), include_groups=False)
by_bin["lesions"] = separation.groupby("bin", observed=True).size()
display(by_bin.reindex(CLASS_LABELS).round(4))
per_phantom = separation.groupby("study_id").apply(lambda g: pd.Series(dict(lesions=len(g), bias=g.spill_separated.sum() / g.tiac_true.sum() - 1)), include_groups=False)
print("Separation bias per phantom (spill map, TIA-weighted): median", round(per_phantom.bias.median(), 4), "; by number of lesions:")
display(per_phantom.groupby(pd.cut(per_phantom.lesions, [0, 3, 10, 30, 1000])).bias.describe()[["count", "50%", "max"]].round(4))


In [ ]:
examples = [s for s in ("psma_01a52e26ce5b5e26_2016-11-21", "psma_505ebb983b71c700_2015-11-07") if s in assignment] + [s for s, v in sorted(assignment.items()) if v == "val"][:1]
examples = examples or train_ids[:2]
cmap = plt.get_cmap("tab20", N.LESION_CODE + 1)
fig, axes = plt.subplots(len(examples), 5, figsize=(13, 4.2 * len(examples)), squeeze=False)
for row, s in zip(axes, examples):
    arrays = N.read_shard(OUT / "shards" / where[s], studies={s})[s]
    src = sources_tv[sources_tv.study_id == s]
    n_les = int(src.lesion_id.max()) if (src.code == N.LESION_CODE).any() else 0
    wc, wl = N.source_sum_tables(src, n_les)
    g = params_train_all[(params_train_all.virtual_id == f"{s}_d00") & (params_train_all.arm == "main") & (params_train_all.scenario == "A")]
    r = N.rebuild(arrays["compartment"], arrays["lesion"].astype(np.int64), arrays["weight"], wc, wl, *N.parameter_arrays(g, n_les))
    kidney = np.argwhere(arrays["compartment"] == N.COMPARTMENT_CODES["kidneys_functional"])
    y = int(np.median(kidney[:, 1])) if len(kidney) else arrays["pet"].shape[1] // 2
    view = lambda a: np.rot90(a[:, y, :])
    pet = np.expm1(arrays["pet"].astype(np.float32)) * N.PET_REF
    a0_ml = np.where(r["inside"], r["a0"] / (GRID_MM ** 3 / 1000), np.nan)
    panels = [(view(pet), "PET (%IA/mL)", "gray_r", dict(vmax=np.percentile(pet, 99.7))), (view(arrays["ct"].astype(np.float32) * N.CT_SCALE), "CT (HU)", "gray", dict(vmin=-200, vmax=400)),
              (np.ma.masked_equal(view(arrays["compartment"]), 0), "compartment code", cmap, dict(vmin=-0.5, vmax=N.LESION_CODE + 0.5, interpolation="nearest")),
              (view(np.log10(np.where(a0_ml > 0, a0_ml, np.nan))), "log10 A0 (fraction IA / mL)", "magma", {}), (view(np.where(r["inside"], np.log10(np.maximum(r["lambda_bio"], N.LAMBDA_BIO_FLOOR)), np.nan)), "log10 lambda_bio (1/h)", "viridis", {})]
    for ax, (img, title, cm, kw) in zip(row, panels):
        im = ax.imshow(img, cmap=cm, aspect=1, **kw)
        ax.set_title(title, loc="left", fontsize=8)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.035)
    row[0].set_title(f"{s}\n{assignment[s]}, d00, scenario A; PET (%IA/mL)", loc="left", fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "examples.png", dpi=130)

## 8. README and the dose placeholder

The README documents every file, field and unit, the target kinds, the grid and the **lesion evaluation rule**. Its numbers are filled from the outputs above. **Dose maps** will come from notebook 21. The place left for them: per phantom, one field `dose_<source>` per Monte Carlo source on the same grid, in Gy per MBq h of that source (unit TIA), so the dose of any draw is $A_{inj} \sum_s \mathrm{TIAC}_s D_s(x)$, the same compact scheme as the targets. Lesion doses stay on the native grid (nb21). `phantoms.parquet` has `dose_available = False` until then.


In [ ]:
import shutil
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    shutil.copy(ROOT / "src" / "net_data.py", tmp)
    probe = "import net_data as N, numpy as np; assert N.M is None; print('net_data alone: OK,', len(N.SOURCE_COMPARTMENTS), 'compartments, lesion code', N.LESION_CODE)"
    out = subprocess.run([sys.executable, "-I", "-c", f"import sys; sys.path.insert(0, {tmp!r}); " + probe], capture_output=True, text=True, cwd=tmp)
COLAB_CHECK = out.stdout.strip() or out.stderr.strip()[-300:]
print(COLAB_CHECK)
assert out.returncode == 0


In [ ]:
def fmt_pct(x):
    return f"{100 * x:.1f}%"


loss = grid_loss.set_index(["grid_mm", "rule"]).tia_weighted_loss_all
ret4 = lesion_bins[(lesion_bins.grid_mm == GRID_MM) & (lesion_bins.rule == RULE)].set_index("bin").retained_median
comp4 = comp_table[(comp_table.grid_mm == GRID_MM) & (comp_table.rule == RULE)].set_index("source").retained_median
stored = memory.groupby("grid_mm").stored_MB.sum() / 1e3
dil = dilation_summary.set_index("r_vox")
sep = separation_summary
kinds = validity.reindex(ORDER[:-1] + ["lesion"])
kind_rows = "\n".join(f"| {s} | {k.target_kind} | {fmt_pct(k.median_abs_curve_error)} |" for s, k in kinds.iterrows())
import importlib.metadata
import platform
python_version = platform.python_version()
package_versions = ", ".join(f"`{p}` {importlib.metadata.version(p)}" for p in ("numpy", "pandas", "scipy", "pyarrow"))
README = f"""# netdata: training data for the network (notebook 22)

Built by `notebooks/06_netdata/22_network_data.ipynb` with `src/net_data.py` (commit {COMMIT}). Source of the kinetics: the frozen virtual cohort (`data/processed/psma/cohort/`). Source of the spatial weights: `psma_mc.source_weights_v2` and `psma_mc.lesion_weights`, the Monte Carlo sources. Licence: derived from PSMA-PET-CT-Lesions (FDAT, CC BY-NC 4.0); do not redistribute outside that licence.

**Test split kept apart:** every file ending in `_test` and `shards/test_*.npz`. Do not use them for training, model selection or any tuning.

## Grid

Isotropic {GRID_MM:g} mm, cropped to the body box plus one voxel, anchored at the corner of the native PET volume (native grids 2.73 x 2.73 x 3.27, 4.07 x 4.07 x 5.0 and 4.07 x 4.07 x 2.0 mm). Label rule `{RULE}`: a voxel is body if at least half of it is body; inside, the source with the largest volume, except where the source with the largest PET activity is a lesion (then that lesion). A lesion that wins no voxel gets its voxel of largest weight (`rescued` in `sources`). Array axes are those of the phantom NIfTI (LAS, index i, j, k); `affine` in `phantoms.parquet` maps (i, j, k) to scanner mm.

Why {GRID_MM:g} mm and not 3 mm: the training phantoms take {stored[GRID_MM]:.1f} GB as stored arrays against {stored[3.0]:.1f} GB at 3 mm (median {memory[memory.grid_mm == GRID_MM].voxels_M.median():.1f} M voxels per phantom), with float16 errors below 0.2% on the PET and 0.5 HU on the CT. The price is on small lesions: the labels keep a median {fmt_pct(ret4['<0.5'])} of the weight of a lesion below 0.5 mL, {fmt_pct(ret4['1-2'])} at 1-2 mL, {fmt_pct(ret4['>=50'])} at 50 mL or more; {fmt_pct(loss[(GRID_MM, RULE)])} of the lesion TIA falls outside its own labels before renormalisation ({fmt_pct(loss[(3.0, RULE)])} at 3 mm; rule `volume` {fmt_pct(loss[(GRID_MM, 'volume')])}). Compartments keep a median {fmt_pct(comp4.min())} ({comp4.idxmin()}) to {fmt_pct(comp4.max())} ({comp4.idxmax()}). Each source is renormalised over its own voxels, so source totals are exact; what the grid loses is position. Hence the lesion rule below.

## Lesion evaluation rule

- **Kinetic quantities per lesion** are not read from the {GRID_MM:g} mm labels as they are. Predicted voxel values are summarised on the lesion voxels **dilated by r = {R_EVAL} voxel ({R_EVAL * GRID_MM:g} mm, ball)**: A0, late and early TIA summed, lambda_bio and lambda_u as A0-weighted means. This radius is the smallest that recovers the spill-out: the share of lesion TIA outside the labels goes from {fmt_pct(dil.tia_weighted_loss[0])} (r = 0) to {fmt_pct(dil.tia_weighted_loss[R_EVAL])} (r = {R_EVAL}); lesion weight retained median {dil.retained_median[R_EVAL]:.3f}, 10th percentile {dil.retained_q10[R_EVAL]:.3f} ({len(dilation)} training lesions, draw 0, scenario A).
- The dilated ring also holds other sources: their TIA there is a median {fmt_pct(dil.other_tia_over_lesion_median[R_EVAL])} of the lesion TIA ({fmt_pct(dil.other_tia_over_lesion_weighted[R_EVAL])} TIA-weighted). So the target is summarised **on the same dilated mask** (rebuild, then the same sums), and the comparison is like with like; the lesion TIAC alone is the cohort value.
- **Lesion doses are computed on the native PET grid**, never on this grid: the TIA of each single lesion through the per-lesion split native kernel of notebook 21 (`cohort_doses.PhantomBasis.lesion_map`), mass-weighted mean per lesion (`PhantomBasis.summarise`).
- **The TIA of a single lesion for the dose is separated, not the dilated sum.** The dilated sum holds other sources' TIA (median {fmt_pct(dil.other_tia_over_lesion_median[R_EVAL])} of the lesion TIA), which must not enter the lesion dose. `net_data.separate_lesion(tia_map, compartment, lesion, i, r_eval={R_EVAL})`: sum of the voxel TIA map over the dilated mask (voxels of other lesions left out), minus, for each ring voxel, the median of the map over the voxels of its own source in a {N.SHELL_VOXELS}-voxel shell just outside the dilated mask (lesion voxels left out). For the true target the lesion TIAC is known exactly (sum over its own labels = cohort value); the separation is for predictions. Residual bias, {len(separation)} training and validation lesions, draw 0, scenario A: separated over true TIAC median {sep.loc['spill, separated', 'median']:.3f} ({sep.loc['spill, separated', 'tia_weighted']:.3f} TIA-weighted, 10-90th percentile {sep.loc['spill, separated', 'q10']:.3f}-{sep.loc['spill, separated', 'q90']:.3f}) when the lesion spills into its neighbours as in the PET, against {sep.loc['spill, plain sum', 'median']:.3f} for the plain dilated sum; on the PBPK-fit baseline, separated over its own lesion TIAC {sep.loc['baseline, separated / its own TIAC', 'median']:.3f} ({sep.loc['baseline, separated / its own TIAC', 'tia_weighted']:.3f}). The bias is positive even when each lesion stays inside its own labels ({sep.loc['label, separated', 'median']:.3f} median), which points to the background: the organ weights are PET-weighted, so the organ voxels next to a lesion carry part of its PET halo and the ring is hotter than the shell (interpretation, not tested further). Report it with lesion doses; it is not corrected.

## Targets: kinds by structure

The TIA of the uptake-washout curve A0 (1/lambda_w - 1/lambda_u) (1/lambda_u = 0 when degenerate) is compared with the PBPK TIAC (`curve_error`). Median |error|, main arm, scenario A, training part: at most {N.CURVE_TOLERANCE:.0%} = `curve` (targets A0, lambda_bio, lambda_u where finite, early TIA); up to {N.EARLY_DOMINANT:.0%} = `late_early` (A0, lambda_bio, early TIA; lambda_u is not a target); above = `tia` (TIAC primary, with early and late TIA; A0 and lambda_bio describe the late part only).

| structure | target kind | median abs curve error |
|---|---|---|
{kind_rows}

Late TIA = A0 / lambda_w, early TIA = TIAC - late TIA (signed: negative with an uptake phase). The voxel TIA is late + early, so it closes on the TIAC for every kind.

## Predictive intervals

A method that estimates a mean and an SD from n draws (the PBPK baseline: the other 9 draws of the phantom; the population prior: the training rows of the structure) gives the column `<q>_n`. Its predictive interval for a new draw is Student t with n - 1 degrees of freedom and scale SD sqrt(1 + 1/n) (`net_data.predictive_halfwidth`; 8 degrees of freedom for 9 draws). This is the correct predictive interval, not a widening. A method that predicts a distribution directly (e.g. a network head with mean and log-variance) leaves `<q>_n` out and is scored as Gaussian. `net_data.evaluate` applies both rules.

## Colab: what to upload besides the shards and parquet

- **Code: `src/net_data.py` only.** It imports `psma_mc` only if present (`M = None` otherwise); everything the training needs (`read_shard`, `source_sum_tables`, `parameter_arrays`, `rebuild`, `unit_weights`, `separate_lesion`, `predictive_halfwidth`, `evaluate`, the constants) runs without it. Building the data (`native_sources`, `prepare`, `cache_one`, `lesion_dilation`, `lesion_separation`) needs `psma_mc.py`, `psma_phantoms.py`, `psma_data.py`, `src/psma_exclusions.py` and the phantoms on disk: local only. Checked by importing `net_data.py` alone in an empty folder ({COLAB_CHECK}).
- **Lesion doses** need `src/cohort_doses.py` (notebook 21) with its own imports and the native phantoms: not part of the Colab upload until notebook 21 says otherwise.
- **Python packages**, versions used to build and check here (Python {python_version}): {package_versions}. On Colab add `torch` (the rebuild accepts torch tensors; not run locally). Parquet is read with `pyarrow`. Only these versions were used; reading the parquet files with an older pandas on Colab was not tested.

## Shards: `shards/<split>_NN.npz`

{N.PHANTOMS_PER_SHARD} phantoms per shard; members `<study_id>/<field>.npy`. Read with `net_data.read_shard(path)` or `np.load(path)["<study_id>/<field>"]`.

| field | dtype | meaning | unit / decoding |
|---|---|---|---|
| `pet` | float16 | 68Ga-PSMA-11 PET, decay-corrected | log1p(PET / {N.PET_REF:g}); PET in %IA/mL = expm1(pet) x {N.PET_REF:g} |
| `ct` | float16 | CT on the PET grid | clip(HU, {N.CT_CLIP[0]:g}, {N.CT_CLIP[1]:g}) / {N.CT_SCALE:g}; HU = ct x {N.CT_SCALE:g} |
| `compartment` | uint8 | source code, table below; 0 = outside the body | |
| `lesion` | uint16 | lesion id (cohort lesion j = id j + 1), 0 = not a lesion | |
| `organs` | uint8 | anatomical label of the phantom (1-15, `phantoms/labelmap.json`), majority by volume, 0 outside the body | for evaluation masks (e.g. L2-L4 = 12) |
| `weight` | float16 | relative weight of the voxel within its source | unit-sum weight w = weight / `weight_sum` of the source (`sources`) |

Compartment codes (labels of `organs.nii.gz` in brackets):

| code | source | phantom labels |
|---|---|---|
""" + "\n".join(f"| {v} | {k} | {', '.join(str(x) for x in N.M.COMPARTMENTS_V2.get(k, [15]))} |" for k, v in N.COMPARTMENT_CODES.items()) + f"""

## Voxel targets of one draw

`net_data.parameter_arrays(params_of_draw, n_lesions)` gives two lookup tables (by compartment code and by lesion id) with columns a0, lambda_bio, lambda_u, tiac_early_h. `net_data.rebuild(compartment, lesion, weight, weight_sum_comp, weight_sum_lesion, comp_table, lesion_table, xp=np or torch)` returns:

- `w`: unit-sum weight per source;
- `a0` = A0_s w(x): fraction of the injected activity in the voxel at t = 0 of the late exponential (divide by the voxel volume, {GRID_MM:g}^3 / 1000 mL, for per-mL);
- `lambda_bio` (1/h) and `lambda_u` (1/h, inf if degenerate), from the label; lambda_u is a target only where `lu_target`;
- `tia_late` = a0 / lambda_w, `tia_early` = tiac_early_h w(x), `tia` = late + early, in h as a fraction of injected activity (multiply by the injected activity for MBq h); lambda_w = lambda_bio + lambda_phys, lambda_phys = {N.LAMBDA_PHYS_H:.6g} /h (177Lu, 6.647 d);
- `degenerate`, `inside` masks.

With PyTorch, pass the index maps as long tensors (uint8 indexing means a boolean mask in torch). The torch path uses the same operations and was not run locally (no torch in `.venv`).

**Excluded source:** the bladder of `psma_3959d1c381a5bcd6_2015-11-12` (training). Its PET is cropped at k >= 84 and the bladder lies below the crop: no voxels on the grid and no Monte Carlo bladder source (226 bladder sources for 227 phantoms). The cohort still gives it a TIAC: `has_source = False`, left out of the rebuild, the baselines and `evaluate`.

## Parquet tables

- `phantoms.parquet` / `phantoms_test.parquet`: one row per phantom: `study_id`, `subject`, `net_split` (train, val, test), `shard`, `scanner`, `grid_mm`, `rule`, `shape`, `native_spacing_mm`, `native_box` (native index slices of the crop), `offset_vox` (offset of the crop in the target grid), `affine` (JSON), `n_lesions`, `dose_available`; training and validation only: `tlv_ml`, `stratum`.
- `sources.parquet` / `sources_test.parquet`: one row per phantom and source: `source` (compartment name or `lesion_<j>`), `code`, `lesion_id`, `n_voxels` and `volume_ml` on the grid, `native_volume_ml`, `weight_sum` (needed by the rebuild), `rescued`, `status` (`ok`, `absent`, `zero_pet`, `uniform_fallback`); training and validation only: `retained` (share of the source weight in its own voxels before renormalisation), `vanished`.
- `params.parquet` / `params_test.parquet`: one row per virtual patient, scenario and source: keys `net_split`, `split` (cohort split), `arm` (main, control), `virtual_id`, `study_id`, `draw`, `scenario` (A, B±0.3, B±0.6), `source`, `structure`; `target_kind`, `has_source`; cohort values `a0` (fraction of IA), `lw`, `lu` (1/h), `tiac_h`, `tiac_early_h`, `tiac_late_h` (h); parametrisation `lambda_bio` (1/h), `lambda_bio_floored` (below {N.LAMBDA_BIO_FLOOR:g} /h; the log uses the floor), `lu_degenerate`, `lu_target`, `log_a0`, `log_lambda_bio`, `log_lambda_u` (NaN if degenerate), `log_tiac`, `log_tiac_late`, `early_share` = tiac_early_h / tiac_h; `tiac_curve_h` = a0 (1/lw − 1/lu), `curve_error` = tiac_curve_h / tiac_h − 1.
- `population_prior.parquet`: per arm and structure, from the training part only (validation excluded), scenario A: `target_kind`, mean and SD of `log_a0_per_ml`, `log_tiac_per_ml`, `log_tiac_late_per_ml`, `log_lambda_bio`, `log_lambda_u` (lambda_u targets only), `early_share`; `p_degenerate`.
- `baseline_population[_test].parquet`, `baseline_pbpk_fit[_test].parquet`: per (`arm`, `virtual_id`, `source`): `<q>_mean`, `<q>_sd`, `<q>_n` (number of draws) for q in `log_a0`, `log_lambda_bio`, `log_lambda_u`, `log_tiac`, `log_tiac_late` (natural log of fraction of IA, 1/h, h) and `early_share` (linear); `target_kind`, `p_degenerate`. The PBPK baseline is the other 9 scenario-A draws of the same phantom (main arm only) and applies to every scenario. Score any prediction with `net_data.evaluate(prediction, params)`.
- `target_distributions.csv`, `figures/`: what notebook 22 reports (`figures/rebuild_check.parquet`: the rebuild check of section 7).

## Dose (placeholder)

Not yet here: dose maps come from notebook 21. Planned: per phantom, `dose_<source>` on the same grid in Gy per MBq h of that source, dose of a draw = injected activity x sum_s tiac_s x D_s. Lesion doses on the native grid (rule above). `dose_available` in `phantoms.parquet` will say when.
"""
(OUT / "README.md").write_text(README)
print(README[README.index("## Grid"):README.index("## Shards")])


## Summary

- **Data**: 227 phantoms (153 train, 27 validation, 47 test kept apart), isotropic 4 mm, label rule `hybrid`, stored once per phantom. Per draw and scenario there is only a parameter table; `net_data.rebuild` turns it into voxel maps (NumPy or torch). Total 1.90 GB, of which 1.84 GB shards (train 1.22, validation 0.22, test 0.40).
- **Targets by kind**, decided on the curve error: `curve` for lesions and glands ($A_0$, $\lambda_{bio}$, $\lambda_u$ where finite, early TIA); `late_early` for kidneys, liver, spleen ($A_0$, $\lambda_{bio}$, early TIA); `tia` for bladder, GI, blood pool, spine, rest of body (TIAC, early, late). The voxel TIA = late + early closes on the cohort TIAC exactly for every source (max error 4e-12).
- **Excluded and declared**: the bladder of `psma_3959d1c381a5bcd6_2015-11-12`, cut off by the PET crop, with no voxels and no Monte Carlo source.
- **Lesion evaluation rule**: summaries on the lesion voxels dilated by 1 voxel (4 mm; 0.24% of the lesion TIA left outside against 7.2%), target and prediction on the same mask. For the dose, the TIA of each single lesion is separated from the other sources in the ring by local background subtraction (`separate_lesion`). Residual bias over 4297 lesions with PET-like spill-out: +1.7% median (+0.9% TIA-weighted, 10-90th percentile -2.7% to +9.1%), against +6.7% for the plain dilated sum; +1.9% on the PBPK baseline prediction; per phantom up to +4.3% with more than 10 lesions. Lesion doses on the native grid with the nb21 per-lesion split kernel.
- **Limits of the targets**: in scenario A the PET carries no information on $\lambda_{bio}$, by construction; a large share of the spread of $A_0$ is between draws of the same PET; the cohort has almost no lesion with T_eff near T_phys.
- **Baselines**: the PBPK fit (the generator) beats the population on liver, spleen, glands, bladder and lesions, not on kidneys and not on $\lambda_{bio}$. With Student-t predictive intervals (8 degrees of freedom) it covers 0.926-0.978 at 95% and 0.652-0.726 at 68%.
- **Colab**: `src/net_data.py` alone (imports checked in an empty folder), numpy, pandas, scipy, pyarrow and torch; the README lists the versions.
- **Not done here**: doses (nb21), network design and training, any upload to Drive.